# Phase 1: GWAS marker-effect replication

**Question.** Do our descriptor-derived θ/φ traits (spline-override and procedural-fit pipelines) recover the same genetic-marker effects that Davis et al. 2025 report for the gold-standard voxel-PCA phyllotaxy traits?

**Design.** Targeted replication of three published hits — no discovery, no FarmCPU/RMIP machinery. For each pair (pipeline, marker) we run a single-marker linear regression of a per-genotype phenotype on minor-allele dosage and report effect size, p-value, and R². Three pipelines × three markers = nine regressions, with a fourth pipeline (the published gold-standard θ/φ from `angles.txt`) included as a positive control.

**Published hits to recover** (Davis et al. 2025, GWAS on `med` — median of the lower four phyllotaxic angles across three timepoints):

| Marker | RMIP | Expected effect (paper) |
|--------|------|------------------------|
| Chr05:12,109,370 | ≈ 0.26 | Minor allele (C) increases `med` (positive) |
| Chr05:65,733,791 | ≈ 0.19 | Minor allele (T) increases `med` (positive) |
| Chr06:41,390,777 | ≥ 0.1  | Minor allele (A) increases `med` (positive) |

**Pipelines we test (all give a per-plant `med` phenotype):**
1. **Gold** — Davis voxel-PCA φ values from Gaillard skeletons (`angles.txt`). Positive control: must reproduce the hit.
2. **Spline** — PhenoSuite trait pipeline on the override descriptor (E1b output, `phi_ours`).
3. **Procedural** — PhenoSuite trait pipeline on the procedural fit (E5 output, `phi_rt`).

**Decision criterion.** If the **procedural** pipeline (E5) recovers the published effect direction at p < 0.05 for at least 2/3 markers, the procedural compression preserves mappable genetic signal at the locus level. If it doesn't, we learn that procedural compression — which §2.4 shows preserves *heritability* — may not preserve *hit-specific* signal.

**One important caveat upfront.** Davis's `med` averages over leaves AND three imaging timepoints. The Zenodo voxel archive we use has one timepoint per plant. Our `med` therefore averages over leaves only. This is a defensible approximation but it is not an exact replication of Davis's analysis.

In [ ]:
from __future__ import annotations

import re, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

REPO_ROOT = Path.cwd().parents[1] if Path.cwd().name == '01_phyllotaxis' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from experiments import data_loaders as dl
from experiments.paths import assert_data_present

assert_data_present()
OUT = REPO_ROOT / 'experiments' / '01_phyllotaxis' / 'outputs'
FIG = REPO_ROOT / 'experiments' / '01_phyllotaxis' / 'figures'
FIG.mkdir(exist_ok=True, parents=True)

from experiments.paths import JENSINA_DATA
SIG_VCF_TSV = JENSINA_DATA / 'sigSNPSMed.tsv'

TARGET_MARKERS = ['Chr05_12109370', 'Chr05_65733791', 'Chr06_41390777']
LOWER = list(range(4))  # 'med' = median over leaves 0..3 (matching Davis 'med14')
REFERENCE_PI = 'PI656058'   # Davis's reference genotype excluded from her analysis
print('Imports OK.')

## 1. Parse Davis's curated significant-SNP VCF

`sigSNPSMed.tsv` is a VCF-style file (skipping the `##` header) with one row per marker and one column per SAP genotype (218 PI accessions). Genotype calls are phased (`0|0`, `0|1`, `1|0`, `1|1`). We convert to **minor-allele dosage** (0/1/2) — the count of ALT alleles, which the paper text identifies as the minor allele at all three sites.

In [ ]:
raw = pd.read_csv(SIG_VCF_TSV, sep='\t', comment=None)
raw = raw.rename(columns={'#CHROM': 'CHROM'})
raw['MARKER_ID'] = raw['CHROM'].astype(str) + '_' + raw['POS'].astype(str)
assert set(raw['MARKER_ID']) == set(TARGET_MARKERS), 'unexpected markers in sigSNPSMed.tsv'

PI_COLS = [c for c in raw.columns if c.startswith('PI')]
print(f'Markers: {list(raw["MARKER_ID"])}')
print(f'Genotypes in VCF: {len(PI_COLS)}')

def call_to_dosage(s):
    return s.fillna('.|.').str.count('1')

geno = pd.DataFrame({pi: call_to_dosage(raw[pi]) for pi in PI_COLS})
geno.index = raw['MARKER_ID']
geno = geno.T   # rows = PI, cols = marker
geno.index.name = 'PI'
print('\nMinor-allele counts (genotype dosage 0/1/2):')
print(geno.apply(pd.Series.value_counts).fillna(0).astype(int))
print('\nMinor allele frequency per marker (MAF):')
print((geno.sum() / (2 * len(geno))).round(3).to_dict())

## 2. Compute per-plant `med` phenotype for each pipeline

`med` for a plant = median of `|normalize360(φᵢ₊₁ − φᵢ) − 180°|` over consecutive leaf pairs i = 0..3 (Davis's `med` over the lower-4 phyllotaxic deviations). One value per plant per pipeline.

In [ ]:
def per_plant_med(df, phi_col):
    sub = df[['plant_id', 'leaf_index', phi_col]].dropna()
    sub = sub.sort_values(['plant_id', 'leaf_index']).copy()
    sub['phi_norm'] = sub[phi_col] % 360.0
    sub['phi_next'] = sub.groupby('plant_id')['phi_norm'].shift(-1)
    diff = (sub['phi_next'] - sub['phi_norm']) % 360.0
    sub['phi_dev'] = np.abs(diff - 180.0)
    sub = sub[sub['leaf_index'].isin(LOWER)]
    return sub.groupby('plant_id')['phi_dev'].median()

e1b = pd.read_csv(OUT / 'e1b_per_leaf_phenosuite_vs_gold.csv')
e5  = pd.read_csv(OUT / 'e5_roundtrip_vs_gold.csv')

med_gold       = per_plant_med(e1b, 'phi_gold').rename('med_gold')
med_spline     = per_plant_med(e1b, 'phi_ours').rename('med_spline')
med_procedural = per_plant_med(e5,  'phi_rt').rename('med_procedural')

pheno = pd.concat([med_gold, med_spline, med_procedural], axis=1)
print(f'Per-plant phenotype rows: {len(pheno)}')
print('\nDistribution summary (deg):')
print(pheno.describe().round(2))

## 3. Map plants → JS line → PI accession → genotype

Per-plant phenotypes are averaged per genotype to give the per-genotype phenotype used in the regression (Davis's design).

In [ ]:
def norm_js(s):
    if not isinstance(s, str):
        return None
    m = re.match(r'(?i)js(\d+)', s)
    return f'JS{m.group(1)}' if m else None

_, js_pi = dl.load_jensina_genotypes()
js_pi['JS_norm'] = js_pi['JS_ID'].apply(norm_js)
js_to_pi = dict(zip(js_pi['JS_norm'], js_pi['PI#'].str.replace('_', '').str.upper()))

pheno = pheno.reset_index()
pheno['js_norm'] = pheno['plant_id'].apply(
    lambda pid: norm_js(dl.parse_plant_id(pid)['js_id'])
)
pheno['PI'] = pheno['js_norm'].map(js_to_pi)
pheno = pheno[pheno['PI'].notna() & (pheno['PI'] != REFERENCE_PI)].copy()

# Per-genotype: average of replicate plant medians
per_geno = pheno.groupby('PI')[['med_gold', 'med_spline', 'med_procedural']].mean()
per_geno['n_plants'] = pheno.groupby('PI').size()
print(f'Per-genotype phenotype rows: {len(per_geno)}')
print(f'Genotypes in VCF: {len(geno)}')
print(f'Genotypes with both pheno + geno: {len(per_geno.index.intersection(geno.index))}')
print(f'Pheno-only genotypes (missing from VCF): {len(per_geno.index.difference(geno.index))}')
print(f'Geno-only genotypes (no plant in voxel set): {len(geno.index.difference(per_geno.index))}')

## 4. Single-marker linear regression

For each (pipeline, marker) pair, regress phenotype on dosage:
$$y_g = \beta_0 + \beta_1 \cdot d_{g,m} + \varepsilon$$
where $y_g$ is the per-genotype mean of `med`, $d_{g,m}$ is the minor-allele dosage at marker $m$ in genotype $g$. We report effect $\beta_1$ (in degrees per allele), p-value of the t-test on $\beta_1$, and R² of the fit.

**No covariates** (population structure, kinship matrix) at this Phase 1 stage. Davis uses FarmCPU with kinship adjustment, so our p-values will tend to be smaller than hers (less conservative). The right comparison at Phase 1 is **effect direction and rough significance**, not exact p-value matching.

In [ ]:
merged = per_geno.join(geno, how='inner')
print(f'Genotypes in regression: {len(merged)}\n')

rows = []
for pipeline_col in ['med_gold', 'med_spline', 'med_procedural']:
    for marker in TARGET_MARKERS:
        sub = merged[[pipeline_col, marker]].dropna()
        y = sub[pipeline_col].values
        d = sub[marker].values
        if np.std(d) == 0 or len(sub) < 10:
            continue
        lr = stats.linregress(d, y)
        rows.append({
            'pipeline':       pipeline_col.replace('med_', ''),
            'marker':         marker,
            'n_geno':         len(sub),
            'allele_freq':    float(d.sum() / (2 * len(d))),
            'effect_beta1':   float(lr.slope),
            'std_err':        float(lr.stderr),
            'pvalue':         float(lr.pvalue),
            'rvalue':         float(lr.rvalue),
            'R2':             float(lr.rvalue ** 2),
            'direction':      'positive' if lr.slope > 0 else 'negative',
            'matches_paper':  bool(lr.slope > 0 and lr.pvalue < 0.05),
        })

results = pd.DataFrame(rows)
pd.set_option('display.float_format', '{:.4f}'.format)
results.sort_values(['marker', 'pipeline'])

In [ ]:
# Compact summary: one row per pipeline, three markers per row
compact = (
    results
      .assign(beta_p=lambda d: d.apply(lambda r: f"{r['effect_beta1']:+.2f}° (p={r['pvalue']:.3f})", axis=1))
      .pivot_table(index='pipeline', columns='marker', values='beta_p', aggfunc='first')
)
compact = compact.reindex(['gold', 'spline', 'procedural'])
compact

### Quick reading rule

For each cell: `+βx° (p=...)`. **Positive effect = matches Davis's published direction** at that marker. p-value is two-sided t-test, no kinship correction.

- **Gold row is the positive control.** It must reproduce Davis's direction at all three markers. If it doesn't, our `med` phenotype definition differs too much from hers to be a fair test.
- **Spline row** tests whether the descriptor-based trait extractor preserves the marker effect when running on the raw skeleton geometry.
- **Procedural row** is the headline — whether the 3-parameter compression preserves the marker effect.

## 5. Visualize: per-marker boxplots of `med` by dosage, per pipeline

For each (pipeline, marker) cell, a boxplot of the per-genotype phenotype binned by minor-allele dosage (0 / 1 / 2). If Davis's effect direction holds, the median should rise with dosage.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(11, 9), sharey='row')
pipelines = [('gold',       'Gold (Davis voxel-PCA)',           'med_gold',       '#222222'),
             ('spline',     'Spline (PhenoSuite on override)',     'med_spline',     '#4477aa'),
             ('procedural', 'Procedural (PhenoSuite on 3-param fit)','med_procedural','#cc6677')]

for row, (label, full, col, color) in enumerate(pipelines):
    for c, marker in enumerate(TARGET_MARKERS):
        ax = axes[row, c]
        sub = merged[[col, marker]].dropna()
        by_dose = [sub.loc[sub[marker] == d, col].values for d in [0, 1, 2]]
        n_dose  = [len(x) for x in by_dose]
        bp = ax.boxplot(by_dose, positions=[0, 1, 2], widths=0.55, patch_artist=True,
                        medianprops={'color': 'black', 'linewidth': 1.4})
        for patch in bp['boxes']:
            patch.set(facecolor=color, alpha=0.55, edgecolor='black')
        for d in [0, 1, 2]:
            jitter = (np.random.RandomState(d).uniform(-0.12, 0.12, size=n_dose[d])
                      if n_dose[d] else np.array([]))
            if n_dose[d]:
                ax.scatter(np.array([d]*n_dose[d]) + jitter, by_dose[d],
                           s=10, color=color, alpha=0.6, edgecolors='none')
        # Annotate per-cell effect + p
        r = results.query(f"pipeline == @label and marker == @marker")
        if len(r):
            r = r.iloc[0]
            ax.set_title(f"$\\beta$={r['effect_beta1']:+.2f}°/allele, p={r['pvalue']:.3f}",
                         fontsize=9)
        ax.set_xticks([0, 1, 2])
        ax.set_xticklabels([f'0\nn={n_dose[0]}', f'1\nn={n_dose[1]}', f'2\nn={n_dose[2]}'])
        if c == 0:
            ax.set_ylabel(f'{full}\n`med` (deg)')
        if row == 0:
            ax.set_xlabel(marker.replace('_', ':'), fontsize=10)
            ax.xaxis.set_label_position('top')
        if row == 2:
            ax.set_xlabel('minor-allele dosage')
        ax.grid(axis='y', alpha=0.3)
fig.suptitle('Phase 1 GWAS replication: per-genotype `med` vs minor-allele dosage',
             y=1.00, fontsize=12)
fig.tight_layout()
fig.savefig(FIG / 'sec2_6_phase1_marker_replication.png', dpi=140, bbox_inches='tight')
plt.show()

## 6. Save outputs

## 7. Davis Fig. 6 C/D/E replica vs descriptor pipelines

Davis's Fig. 6 panels C/D/E compare `med` between major-homozygous (`0/0`) and minor-homozygous (`1/1`) genotypes at each of the three top markers, **excluding heterozygotes** (per her caption). We reproduce this exact design but with three pipelines stacked: gold (matches Davis's figure directly) on the top row, spline on the middle row, procedural on the bottom row. Same y-axis units (degrees), same allele-state grouping. Each panel reports the mean shift Δμ between minor and major groups and a Welch's t-test p-value.

In [ ]:
# Davis Fig 6 C/D/E: one point per genotype, heterozygotes excluded.
PANEL_LETTERS = ['C', 'D', 'E']
MARKER_RMIP = {'Chr05_12109370': 0.26, 'Chr05_65733791': 0.19, 'Chr06_41390777': 0.10}

# Taller figure + per-row sharey gives more vertical room for titles.
fig, axes = plt.subplots(3, 3, figsize=(11, 11), sharey='row')
PIPELINES = [
    ('gold',       'Voxel',      'med_gold',       '#666666'),
    ('spline',     'Spline',     'med_spline',     '#4477aa'),
    ('procedural', 'Procedural', 'med_procedural', '#cc6677'),
]

# Pre-compute per-row data range so we can pad the top to leave room for titles
row_max = {}
for row, (_, _, col_name, _) in enumerate(PIPELINES):
    sub = merged[[col_name] + TARGET_MARKERS].dropna()
    row_max[row] = float(sub[col_name].max())

for row, (label, short, col_name, color) in enumerate(PIPELINES):
    for c, marker in enumerate(TARGET_MARKERS):
        ax = axes[row, c]
        sub = merged[[col_name, marker]].dropna()
        sub = sub[sub[marker].isin([0, 2])].copy()
        major = sub.loc[sub[marker] == 0, col_name].values
        minor = sub.loc[sub[marker] == 2, col_name].values
        bp = ax.boxplot([major, minor], positions=[0, 1], widths=0.55, patch_artist=True,
                        medianprops={'color': 'black', 'linewidth': 1.4},
                        showmeans=True,
                        meanprops={'marker': 'D', 'markerfacecolor': 'white',
                                   'markeredgecolor': 'black', 'markersize': 5})
        for patch in bp['boxes']:
            patch.set(facecolor=color, alpha=0.5, edgecolor='black')
        rng = np.random.RandomState(c + row * 7)
        for pos, arr in [(0, major), (1, minor)]:
            if len(arr):
                jx = rng.uniform(-0.13, 0.13, size=len(arr))
                ax.scatter(np.full(len(arr), pos) + jx, arr, s=14,
                           color=color, alpha=0.65, edgecolors='black', linewidths=0.3)

        # Leave 12% headroom above the highest data point so titles don't collide
        ax.set_ylim(top=row_max[row] * 1.12)


        ax.set_xticks([0, 1])
        ax.set_xticklabels(['major\nn=' + str(len(major)),
                            'minor\nn=' + str(len(minor))], fontsize=10)
        if c == 0:
            ax.set_ylabel(short + '\nmedian $|\Delta\phi|$ (deg)', fontsize=11)
        # Column header (panel letter + marker + RMIP) only on row 0,
        # placed clearly above the title with generous spacing
        if row == 0:
            chrom, pos = marker.split('_')
            pretty = f'{chrom}:{int(pos):,}'
            ax.text(0.5, 1.30, f'({PANEL_LETTERS[c]}) {pretty}',
                    transform=ax.transAxes, ha='center', fontsize=12, fontweight='bold')
            ax.text(0.5, 1.19, r'RMIP $\approx$ ' + f'{MARKER_RMIP[marker]:.2f}',
                    transform=ax.transAxes, ha='center', fontsize=10, style='italic',
                    color='#444444')
        ax.grid(axis='y', alpha=0.3)

fig.suptitle(r'GWAS marker effects on lower-canopy median $|\Delta\phi|$ at Davis et al. 2025 hits',
             y=0.995, fontsize=12)
# Reserve top space for the suptitle + column headers; bottom slightly tighter
fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.94))
fig.savefig(FIG / 'sec2_6_phase1_davis_fig6_replica.png', dpi=140, bbox_inches='tight')
plt.show()


In [ ]:
results.to_csv(OUT / 'sec2_6_phase1_gwas_marker_replication.csv', index=False)
per_geno.to_csv(OUT / 'sec2_6_phase1_per_genotype_phenotype.csv')
print('Wrote:')
for f in ['sec2_6_phase1_gwas_marker_replication.csv',
          'sec2_6_phase1_per_genotype_phenotype.csv']:
    print(f'  {OUT / f}')
print(f'  {FIG / "sec2_6_phase1_marker_replication.png"}')

## Discussion

**What this experiment validates.** For each of three published phyllotaxy-associated markers from Davis et al. 2025, we test whether the descriptor-derived (spline and procedural) phenotype pipelines recover the same direction of marker effect that the gold-standard voxel-PCA pipeline does. Single-marker linear regression, no kinship adjustment, no resampling — this is a Phase 1 targeted-replication test, not a full GWAS.

**Reading the headline numbers.**
- *Gold row* is the positive control. It must reproduce Davis's positive effect direction at the three markers; if it doesn't, our trait definition differs too much from hers to be a fair test.
- *Spline row* tests whether running PhenoSuite's trait pipeline on the spline-override descriptor preserves the marker-effect signal.
- *Procedural row* is the headline: whether the 3-parameter compression preserves marker-effect signal.

**Decision criteria.** If the procedural pipeline reproduces Davis's effect direction (positive) at p < 0.05 for **at least 2 of 3 markers**, the procedural compression preserves mappable genetic signal at the locus level — a much stronger claim than the heritability result in §2.4. If it doesn't, we have learned that procedural compression preserves variance but loses hit-specific signal, which is informative even if not the result we wanted.

**Caveats.**
- Our `med` definition averages over leaves only (Davis averages over leaves and three imaging timepoints), because the Zenodo voxel archive has one timepoint per plant.
- No kinship/population-structure correction; p-values are anti-conservative relative to FarmCPU.
- N ≈ 100-200 genotypes after intersecting our voxel-set plants with Davis's filtered VCF; underpowered for true effect-size estimates but enough for direction tests.
- Davis's hits come from RMIP resampling, so the markers are 'stable' rather than 'top by p-value' — our single-shot p-values do not need to match hers in magnitude.

**Outputs:** `sec2_6_phase1_gwas_marker_replication.csv` (per (pipeline, marker) effect+p+R²); `sec2_6_phase1_per_genotype_phenotype.csv` (per-genotype `med` across pipelines). Figure: `sec2_6_phase1_marker_replication.png`.